In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("../data/processed/loan_selected.csv")

df.shape

(1303638, 23)

In [4]:
df.shape

(1303638, 23)

In [5]:
df["loan_status"].value_counts()

loan_status
Fully Paid     1041952
Charged Off     261655
Default             31
Name: count, dtype: int64

See Missing Values

In [6]:
df["default"].value_counts()

default
0    1041952
1     261686
Name: count, dtype: int64

Missing percentage

In [7]:
missing = df.isnull().sum().sort_values(ascending=False)

missing[missing > 0]

emp_length        75457
revol_util          810
dti                 312
inq_last_6mths        1
dtype: int64

Missing percentage

In [8]:
missing_pct = (df.isnull().mean() * 100).sort_values(ascending=False)

missing_pct[missing_pct > 0]

emp_length        5.788187
revol_util        0.062134
dti               0.023933
inq_last_6mths    0.000077
dtype: float64

Dataset types

In [9]:
df.dtypes

loan_amnt                int64
term                    object
int_rate               float64
installment            float64
grade                   object
sub_grade               object
emp_length              object
home_ownership          object
annual_inc             float64
verification_status     object
purpose                 object
addr_state              object
dti                    float64
delinq_2yrs            float64
inq_last_6mths         float64
open_acc               float64
pub_rec                float64
revol_bal                int64
revol_util             float64
total_acc              float64
application_type        object
loan_status             object
default                  int64
dtype: object

In [10]:
for col in df.columns:
    missing = df[col].isnull().sum()
    if missing > 0:
        print(f"{col}: {missing} missing ({missing/len(df)*100:.2f}%)")

emp_length: 75457 missing (5.79%)
dti: 312 missing (0.02%)
inq_last_6mths: 1 missing (0.00%)
revol_util: 810 missing (0.06%)


In [11]:
df.dtypes.value_counts()

object     10
float64    10
int64       3
Name: count, dtype: int64

Handle missing Values

In [12]:
# Make a copy before cleaning
df_clean = df.copy()

# Categorical missing values
df_clean["emp_length"] = df_clean["emp_length"].fillna("Unknown")

# Numerical missing values
for col in ["dti", "inq_last_6mths", "revol_util"]:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

In [13]:
df_clean.isnull().sum()

loan_amnt              0
term                   0
int_rate               0
installment            0
grade                  0
sub_grade              0
emp_length             0
home_ownership         0
annual_inc             0
verification_status    0
purpose                0
addr_state             0
dti                    0
delinq_2yrs            0
inq_last_6mths         0
open_acc               0
pub_rec                0
revol_bal              0
revol_util             0
total_acc              0
application_type       0
loan_status            0
default                0
dtype: int64

In [14]:
df_clean.dtypes

loan_amnt                int64
term                    object
int_rate               float64
installment            float64
grade                   object
sub_grade               object
emp_length              object
home_ownership          object
annual_inc             float64
verification_status     object
purpose                 object
addr_state              object
dti                    float64
delinq_2yrs            float64
inq_last_6mths         float64
open_acc               float64
pub_rec                float64
revol_bal                int64
revol_util             float64
total_acc              float64
application_type        object
loan_status             object
default                  int64
dtype: object

In [15]:
df_clean.dtypes

loan_amnt                int64
term                    object
int_rate               float64
installment            float64
grade                   object
sub_grade               object
emp_length              object
home_ownership          object
annual_inc             float64
verification_status     object
purpose                 object
addr_state              object
dti                    float64
delinq_2yrs            float64
inq_last_6mths         float64
open_acc               float64
pub_rec                float64
revol_bal                int64
revol_util             float64
total_acc              float64
application_type        object
loan_status             object
default                  int64
dtype: object

In [16]:
df_clean["term"] = (
    df_clean["term"]
    .str.extract(r"(\d+)")
    .astype(int)
)

In [17]:
df_clean["term"].value_counts()

term
36    988774
60    314864
Name: count, dtype: int64

emp_length clean

In [18]:
df_clean["emp_length"] = (
    df_clean["emp_length"]
    .replace({
        "< 1 year": "0",
        "10+ years": "10",
        "1 year": "1"
    })
    .str.extract(r"(\d+)")
    .astype(float)
)

In [19]:
df_clean["emp_length"].value_counts(dropna=False).sort_index()

emp_length
0.0     104552
1.0      85678
2.0     117825
3.0     104204
4.0      78033
5.0      81623
6.0      60934
7.0      58148
8.0      59127
9.0      49504
10.0    428553
NaN      75457
Name: count, dtype: int64

In [20]:
df_clean["emp_length"] = df_clean["emp_length"].fillna(-1)

verify

In [21]:
df_clean[[
    "term",
    "emp_length",
    "int_rate",
    "revol_util",
    "dti"
]].dtypes

term            int64
emp_length    float64
int_rate      float64
revol_util    float64
dti           float64
dtype: object

In [22]:
df_clean[[
    "term",
    "emp_length",
    "int_rate",
    "revol_util",
    "dti"
]].head()

,term,emp_length,int_rate,revol_util,dti
0,36,5.0,22.35,37.0,30.46
1,60,0.0,16.14,64.5,50.53
2,36,10.0,7.56,29.9,18.92
3,36,10.0,11.31,15.3,4.64
4,36,3.0,27.27,65.7,12.37


Identify categorical columns

In [23]:
categorical_cols = df_clean.select_dtypes(include=["object"]).columns.tolist()

categorical_cols

['grade',
 'sub_grade',
 'home_ownership',
 'verification_status',
 'purpose',
 'addr_state',
 'application_type',
 'loan_status']

In [24]:
print("Categorical columns:")
for col in categorical_cols:
    print("-", col)

Categorical columns:
- grade
- sub_grade
- home_ownership
- verification_status
- purpose
- addr_state
- application_type
- loan_status


check target

In [25]:
df_clean["default"].value_counts()

default
0    1041952
1     261686
Name: count, dtype: int64

In [26]:
df_clean["default"].value_counts(normalize=True) * 100

default
0    79.926483
1    20.073517
Name: proportion, dtype: float64

loan_status remov

In [27]:
df_clean = df_clean.drop(columns=["loan_status"])

In [28]:
print(df_clean.shape)
print(df_clean.columns.tolist())

(1303638, 22)
['loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'purpose', 'addr_state', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'application_type', 'default']


Grade encode

In [29]:
grade_mapping = {
    "A": 1,
    "B": 2,
    "C": 3,
    "D": 4,
    "E": 5,
    "F": 6,
    "G": 7
}

df_clean["grade"] = df_clean["grade"].map(grade_mapping)

Sub-grade encode

In [30]:
subgrade_mapping = {
    f"{grade}{num}": i
    for i, (grade, num) in enumerate(
        [(g, n) for g in "ABCDEFG" for n in range(1, 6)],
        start=1
    )
}

df_clean["sub_grade"] = df_clean["sub_grade"].map(subgrade_mapping)

In [31]:
df_clean[["grade", "sub_grade"]].head()

,grade,sub_grade
0,4,20
1,3,14
2,1,3
3,2,8
4,5,25


In [32]:
categorical_cols = df_clean.select_dtypes(include=["object"]).columns.tolist()

categorical_cols

['home_ownership',
 'verification_status',
 'purpose',
 'addr_state',
 'application_type']

In [33]:
print(categorical_cols)

['home_ownership', 'verification_status', 'purpose', 'addr_state', 'application_type']


one-hot encoding

In [34]:
categorical_cols = [
    "home_ownership",
    "verification_status",
    "purpose",
    "addr_state",
    "application_type"
]

df_clean = pd.get_dummies(
    df_clean,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)

In [35]:
print("New shape:", df_clean.shape)

New shape: (1303638, 88)


In [36]:
df_clean.head()

,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_length,annual_inc,dti,delinq_2yrs,...,addr_state_TN,addr_state_TX,addr_state_UT,addr_state_VA,addr_state_VT,addr_state_WA,addr_state_WI,addr_state_WV,addr_state_WY,application_type_Joint App
0,30000,36,22.35,1151.16,4,20,5.0,100000.0,30.46,0.0,...,0,0,0,0,0,0,0,0,0,1
1,40000,60,16.14,975.71,3,14,0.0,45000.0,50.53,0.0,...,0,0,0,0,0,0,0,0,0,1
2,20000,36,7.56,622.68,1,3,10.0,100000.0,18.92,0.0,...,0,0,0,0,0,1,0,0,0,1
3,4500,36,11.31,147.99,2,8,10.0,38500.0,4.64,0.0,...,0,1,0,0,0,0,0,0,0,0
4,8425,36,27.27,345.18,5,25,3.0,450000.0,12.37,0.0,...,0,0,0,0,0,0,0,0,0,1


Check everything is numeric

In [37]:
df_clean.dtypes.value_counts()

int64      77
float64    11
Name: count, dtype: int64

In [38]:
print(
    "Remaining categorical columns:",
    df_clean.select_dtypes(include=["object"]).columns.tolist()
)

Remaining categorical columns: []


In [39]:
import numpy as np

print("Infinite values:", np.isinf(df_clean.select_dtypes(include=np.number)).sum().sum())

Infinite values: 0


In [40]:
print("Missing values:", df_clean.isnull().sum().sum())

Missing values: 0


In [41]:
print(df_clean["default"].value_counts())
print()
print(df_clean["default"].value_counts(normalize=True) * 100)

default
0    1041952
1     261686
Name: count, dtype: int64

default
0    79.926483
1    20.073517
Name: proportion, dtype: float64


In [42]:
df_clean.describe().T

,count,mean,std,min,25%,50%,75%,max
loan_amnt,1303638.0,14416.838167,8699.573905,500.00,8000.00,12000.00,20000.00,40000.00
term,1303638.0,41.796652,10.272223,36.00,36.00,36.00,36.00,60.00
int_rate,1303638.0,13.257304,4.760614,5.31,9.75,12.74,15.99,30.99
installment,1303638.0,438.085968,261.064090,4.93,248.82,375.43,580.45,1719.83
grade,1303638.0,2.751501,1.296944,1.00,2.00,3.00,4.00,7.00
...,...,...,...,...,...,...,...,...
addr_state_WA,1303638.0,0.021756,0.145886,0.00,0.00,0.00,0.00,1.00
addr_state_WI,1303638.0,0.013146,0.113901,0.00,0.00,0.00,0.00,1.00
addr_state_WV,1303638.0,0.003640,0.060221,0.00,0.00,0.00,0.00,1.00
addr_state_WY,1303638.0,0.002175,0.046583,0.00,0.00,0.00,0.00,1.00


Before train/test split, we'll separate:

X = features
y = default

Feature Engineering

Loan-to-Income Ratio

In [45]:
df_clean["loan_to_income"] = (
    df_clean["loan_amnt"] / (df_clean["annual_inc"] + 1)
)

Installment-to-Income Ratio

In [44]:
df_clean["installment_to_income"] = (
    df_clean["installment"] / (df_clean["annual_inc"] + 1)
)

Credit History Density

In [46]:
df_clean["open_to_total_accounts"] = (
    df_clean["open_acc"] / (df_clean["total_acc"] + 1)
)

Delinquency + Public Record Risk

In [47]:
df_clean["credit_issue_count"] = (
    df_clean["delinq_2yrs"] +
    df_clean["pub_rec"]
)

New features check karo

In [48]:
new_features = [
    "loan_to_income",
    "installment_to_income",
    "open_to_total_accounts",
    "credit_issue_count"
]

df_clean[new_features].describe().T

,count,mean,std,min,25%,50%,75%,max
loan_to_income,1303638.0,4.201286,303.902087,0.000171,0.124997,0.199995,0.290906,40000.000000
installment_to_income,1303638.0,0.123006,8.809086,0.000006,0.003865,0.006023,0.008790,1466.850000
open_to_total_accounts,1303638.0,0.474817,0.159695,0.000000,0.357143,0.461538,0.578947,1.555556
credit_issue_count,1303638.0,0.533144,1.054464,0.000000,0.000000,0.000000,1.000000,86.000000


In [49]:
df_clean[new_features].isnull().sum()

loan_to_income            0
installment_to_income     0
open_to_total_accounts    0
credit_issue_count        0
dtype: int64

In [50]:
df_clean.to_csv(
    "../data/processed/loan_cleaned.csv",
    index=False
)

In [51]:
df_clean.shape

(1303638, 92)

In [52]:
df_clean[new_features].describe().T

,count,mean,std,min,25%,50%,75%,max
loan_to_income,1303638.0,4.201286,303.902087,0.000171,0.124997,0.199995,0.290906,40000.000000
installment_to_income,1303638.0,0.123006,8.809086,0.000006,0.003865,0.006023,0.008790,1466.850000
open_to_total_accounts,1303638.0,0.474817,0.159695,0.000000,0.357143,0.461538,0.578947,1.555556
credit_issue_count,1303638.0,0.533144,1.054464,0.000000,0.000000,0.000000,1.000000,86.000000


In [53]:
df_clean[new_features].isnull().sum()

loan_to_income            0
installment_to_income     0
open_to_total_accounts    0
credit_issue_count        0
dtype: int64